# HiDF patch detector
Choose a GPU runtime. In Google Drive, create `MyDrive/HiDF` and upload `Fake-img/`, `Real-img/`, `hidf_clean_source_manifest.csv` and the new `patch_detector/` folder into it. Open this notebook in Colab. Checkpoints and scores will go to `MyDrive/HiDF_runs`. The first training run downloads the torchvision ImageNet weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess
import sys
import torch
import torchvision
import sklearn
from PIL import Image

ROOT = Path('/content/drive/MyDrive/HiDF')
CODE = ROOT / 'patch_detector'
RUNS = ROOT.parent / 'HiDF_runs'
assert (CODE / 'experiment.py').is_file(), f'Check the upload at {CODE}'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0))

## Stage images on the runtime disk
This copies only the two image folders and manifest. It may take time on the first run. Run it again after reconnecting to a fresh runtime.

In [ ]:
import shutil
DATA = Path('/content/hidf_data')
DATA.mkdir(exist_ok=True)
for name in ['Real-img', 'Fake-img']:
    shutil.copytree(ROOT / name, DATA / name, dirs_exist_ok=True)
shutil.copy2(ROOT / 'hidf_clean_source_manifest.csv', DATA)
sys.path.insert(0, str(CODE))
from data import read_manifest
print({split: len(rows) for split, rows in read_manifest(DATA).items()})

## Train your model
The default trains the proposed model. To run the controlled study, set VARIANTS to all three entries and SEEDS to [42, 43, 44]. Rerunning resumes completed epochs for each configuration. Keep batch size fixed across variants. Each saved validation score uses only the validation split.

In [ ]:
VARIANTS = ['full']  # Controlled study: ['global', 'patch', 'full']
SEEDS = [42]
EPOCHS = 20
BATCH = 16

def experiment(mode, variant, seed):
    subprocess.run([sys.executable, '-u', str(CODE / 'experiment.py'), mode,
                    '--data', str(DATA), '--runs', str(RUNS), '--variant', variant,
                    '--seed', str(seed), '--epochs', str(EPOCHS), '--batch', str(BATCH)], check=True)

for variant in VARIANTS:
    for seed in SEEDS:
        experiment('train', variant, seed)

## Final held-out evaluation
Run this cell after freezing settings and completing all intended training. The code loads each run's highest clean validation AUC checkpoint, then evaluates clean images and three JPEG qualities. The table displays each seed separately.

In [ ]:
import json
import pandas as pd

records = []
for variant in VARIANTS:
    for seed in SEEDS:
        experiment('test', variant, seed)
        result = json.loads((RUNS / f'{variant}-seed{seed}' / 'test.json').read_text())
        for condition, scores in result['metrics'].items():
            records.append(dict(variant=variant, seed=seed, condition=condition, **scores))
display(pd.DataFrame(records).round(4))